# SetFit

In [10]:
!pip install setfit sentence_transformers

Defaulting to user installation because normal site-packages is not writeable
  Using cached setfit-1.1.3-py3-none-any.whl.metadata (12 kB)
  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached transformers-5.10.2-py3-none-any.whl.metadata (33 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached huggingface_hub-1.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.12.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.26.7-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached filelock-3.29.1-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\joachim.querule\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\pkg_resources\\tests\\data\\my-test-package_unpacked-egg\\my_test_package-1.0-py3.7.egg\\EGG-INFO\\dependency_links.txt'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [11]:
# This version works well (so far)
!pip install "transformers>=4.20.0,<5.0.0" "setfit==1.1.3" -q

# You may have to restart session after installing

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\joachim.querule\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\pkg_resources\\tests\\data\\my-test-package_unpacked-egg\\my_test_package-1.0-py3.7.egg\\EGG-INFO\\dependency_links.txt'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [12]:
import sys

print(sys.executable)

c:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\.venv\Scripts\python.exe


In [13]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments  # ← new API
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [14]:
from pathlib import Path
import os

project_dir = Path(
    r"C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\Data"
)

if not project_dir.exists():
    raise FileNotFoundError(
        f"Le dossier n'existe pas : {project_dir}"
    )

os.chdir(project_dir)

print("Dossier de travail :", Path.cwd())

Dossier de travail : C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\Data


In [15]:
from pathlib import Path

print("Dossier actuel :", Path.cwd())

Dossier actuel : C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\Data


In [16]:
from pathlib import Path

csv_files = list(Path.cwd().rglob("*.csv"))

print("Fichiers CSV trouvés :")

for fichier in csv_files:
    print(fichier.resolve())

Fichiers CSV trouvés :
C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\data\Dataset.csv
C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\data\tweets_dataset.csv


In [17]:
from pathlib import Path

file_path = Path(
    r"C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\data\tweets_dataset.csv"
)

with open(
    file_path,
    mode="r",
    encoding="utf-8-sig",
    errors="replace"
) as fichier:
    for numero, ligne in enumerate(fichier, start=1):
        if 30 <= numero <= 35:
            print(numero, repr(ligne))

30 'Billie Eilish,Pain is good,TRUE\n'
31 '"Ariana Grande,""""""Is it bad that I sometimes wanna punch my 4-year-old brother in the face?"""""",TRUE"\n'
32 '"Tyler, the Creator,""""""My therapist told me to stop tweeting and start journaling but now my journal is a Twitter thread"""""",TRUE"\n'
33 '"Conan O Brien, """"""I wonder if Taco Bell has a breakfast menu because I m craving a Chalupa at 8 AM."""""", TRUE"\n'
34 '"Ryan Reynolds,""If I disappear, it’s because I lost a staring contest with my toddler."",FALSE"\n'
35 'Tyler, the Creator,i just saw a goose chase a child and honestly i’m rooting for the goose.,FALSE\n'


In [18]:
import pandas as pd
from pathlib import Path

file_path = Path(
    r"C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\data\tweets_dataset.csv"
)

if not file_path.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {file_path}"
    )

df = pd.read_csv(
    file_path,
    sep=",",
    encoding="utf-8-sig",
    on_bad_lines="skip"
)

print("Dimensions :", df.shape)
print("Colonnes :", df.columns.tolist())

df.head()

Dimensions : (106, 3)
Colonnes : ['name', 'tweet', 'is_real']


,name,tweet,is_real
0,Billie Eilish,i’m fine. just floating in a hoodie-shaped clo...,False
1,Ryan Reynolds,I watched Frozen without my two-year-old this ...,True
2,Billie Eilish,people really be like “you’ve changed” like th...,False
3,Billie Eilish,people really be like “you’ve changed” like th...,False
4,Elon Musk,Nuke Mars!,True


In [19]:
df = pd.read_csv(
    file_path,
    sep=",",
    quotechar='"',
    engine="python",
    encoding="utf-8-sig",
    on_bad_lines="skip"
)

df.head()

,name,tweet,is_real
0,Billie Eilish,i’m fine. just floating in a hoodie-shaped clo...,False
1,Ryan Reynolds,I watched Frozen without my two-year-old this ...,True
2,Billie Eilish,people really be like “you’ve changed” like th...,False
3,Billie Eilish,people really be like “you’ve changed” like th...,False
4,Elon Musk,Nuke Mars!,True


In [20]:
# Rename columns
df = df.rename(columns={'tweet': 'text', 'is_real': 'label'})

# Map True/False in 'label' column to 0/1
df['label'] = df['label'].map({True: 1, False: 0})

df.head()

,name,text,label
0,Billie Eilish,i’m fine. just floating in a hoodie-shaped clo...,0.0
1,Ryan Reynolds,I watched Frozen without my two-year-old this ...,1.0
2,Billie Eilish,people really be like “you’ve changed” like th...,0.0
3,Billie Eilish,people really be like “you’ve changed” like th...,0.0
4,Elon Musk,Nuke Mars!,1.0


In [21]:
print(df['label'].value_counts(normalize=False))

label
1.0    31
0.0    28
Name: count, dtype: int64


In [22]:
# To ensure a balanced proportion of labels, perform stratified sampling for the training set.
# We want 8 texts for training, so we'll aim for 4 of each label if possible.

# Sample 4 texts where label is 0
train_df_label_0 = df[df['label'] == 0].sample(n=4, random_state=42)

# Sample 4 texts where label is 1
train_df_label_1 = df[df['label'] == 1].sample(n=4, random_state=42)

# Concatenate to form the balanced training DataFrame
train_df = pd.concat([train_df_label_0, train_df_label_1])

# Shuffle the training DataFrame to mix the labels
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

# The remaining data (for test and eval)
remaining_df = df.drop(train_df.index)

# Split remaining into test and eval (50/50 split)
test_df = remaining_df.sample(frac=0.5, random_state=42)
eval_df = remaining_df.drop(test_df.index)

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))

# Check sizes
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}, Eval: {len(eval_dataset)}")
print("Train label distribution:")
print(train_df['label'].value_counts(normalize=False))

Train: 8, Test: 49, Eval: 49
Train label distribution:
label
0.0    4
1.0    4
Name: count, dtype: int64


In [23]:
# Prepare the evaluation metrics

def compute_metrics(y_pred, y_test):
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted"
    )
    return {
        "accuracy":  round(accuracy, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
    }

In [24]:
from pathlib import Path
from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Dossier local dans ton projet VS Code
output_dir = Path("setfit_2epochs")
output_dir.mkdir(parents=True, exist_ok=True)

# Charger le modèle depuis Hugging Face
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-mpnet-base-v2"
)

print("Modèle chargé avec succès.")
print("Dossier de sortie :", output_dir.resolve())

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Modèle chargé avec succès.
Dossier de sortie : C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\data\setfit_2epochs


In [25]:
import pandas as pd
from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Chemin du fichier CSV
file_path = Path(
    r"C:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\data\tweets_dataset.csv"
)

# 2. Vérifier que le fichier existe
if not file_path.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {file_path}"
    )

# 3. Charger le fichier dans la variable df
df = pd.read_csv(
    file_path,
    sep=",",
    encoding="utf-8-sig",
    on_bad_lines="skip"
)

# 4. Nettoyer les noms des colonnes
df.columns = df.columns.str.strip().str.lower()

print("Dimensions :", df.shape)
print("Colonnes disponibles :", df.columns.tolist())
print(df.head())

Dimensions : (106, 3)
Colonnes disponibles : ['name', 'tweet', 'is_real']
            name                                              tweet is_real
0  Billie Eilish  i’m fine. just floating in a hoodie-shaped clo...   False
1  Ryan Reynolds  I watched Frozen without my two-year-old this ...    True
2  Billie Eilish  people really be like “you’ve changed” like th...   False
3  Billie Eilish  people really be like “you’ve changed” like th...   False
4      Elon Musk                                         Nuke Mars!    True


In [26]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Nettoyer les noms de colonnes
df.columns = df.columns.str.strip().str.lower()

print("Colonnes disponibles :", df.columns.tolist())

# Garder les colonnes utiles et les renommer
data = (
    df[["tweet", "is_real"]]
    .dropna()
    .rename(
        columns={
            "tweet": "text",
            "is_real": "label"
        }
    )
)

# Convertir TRUE/FALSE ou 1/0 en valeurs numériques
data["label"] = (
    data["label"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "TRUE": 1,
        "FALSE": 0,
        "1": 1,
        "0": 0
    })
)

# Supprimer les labels non reconnus
data = data.dropna(subset=["label"])

# Transformer les labels en entiers
data["label"] = data["label"].astype(int)

print("\nAperçu des données :")
print(data.head())

print("\nRépartition des classes :")
print(data["label"].value_counts())

Colonnes disponibles : ['name', 'tweet', 'is_real']

Aperçu des données :
                                                text  label
0  i’m fine. just floating in a hoodie-shaped clo...      0
1  I watched Frozen without my two-year-old this ...      1
2  people really be like “you’ve changed” like th...      0
3  people really be like “you’ve changed” like th...      0
4                                         Nuke Mars!      1

Répartition des classes :
label
1    31
0    28
Name: count, dtype: int64


In [28]:
train_df, eval_df = train_test_split(
    data,
    train_size=8,
    random_state=42,
    stratify=data["label"]
)

train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True),
    preserve_index=False
)

eval_dataset = Dataset.from_pandas(
    eval_df.reset_index(drop=True),
    preserve_index=False
)

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 8
})
Dataset({
    features: ['text', 'label'],
    num_rows: 51
})


In [29]:
import os
os.environ["WANDB_DISABLED"] = "true" # If you don't want to use W&B

output_dir = "/content/setfit_2epochs" # Create a folder to store the fined-tuned model

# Load SetFit model from Hub
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-mpnet-base-v2")

# Setting
args = TrainingArguments(
    batch_size=16,
    num_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric=compute_metrics,
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate()
print(">>>> Evaluation Result:", metrics)

# Save the model
trainer.model.save_pretrained(output_dir)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Map: 100%|██████████| 8/8 [00:00<00:00, 2590.08 examples/s]
***** Running training *****
  Num unique pairs = 40
  Batch size = 16
  Num epochs = 2
c:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ni

Epoch,Training Loss,Validation Loss
1,0.197900,0.265545
2,0.197900,0.247586


c:\Users\joachim.querule\OneDrive - LYCEE Jules Haag\Bureau\nlp_ning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
***** Running evaluation *****


>>>> Evaluation Result: {'accuracy': 0.6078, 'precision': 0.6497, 'recall': 0.6078, 'f1': 0.591}


# **Understanding the Training Arguments:**

`batch_size=16` — The model learns from 16 examples at a time, rather than all at once.

`num_epochs=2` — It reads through the entire training data 2 times to reinforce learning.

`eval_strategy="epoch"` — After each full read-through, test the model on unseen examples to check its progress.

`save_strategy="epoch"` — Take a snapshot of the model after each test.

`load_best_model_at_end=True` — When training finishes, roll back to whichever snapshot performed best — not necessarily the last one.

---

# **Understanding the values produced during the Training process:**
`Epoch` — Which round you're on out of 4 total.

`Training Loss` — How many mistakes the model makes on the data it's learning from. Lower is better. This should drop steadily each epoch.

`Validation Loss` — How many mistakes it makes on data it's never seen before. This is the more honest score — it tells you if the model truly learned or just memorized.

---

### What to watch for

**✅ Healthy training** — Both losses drop together over epochs.

**⚠️ Overfitting** — Training loss keeps dropping but Validation loss starts *rising* (like epochs 3→4 above). The model is memorizing the training data rather than learning general patterns. This is exactly why `load_best_model_at_end=True` exists.

**⚠️ Underfitting** — Both losses are still high even at the last epoch. The model hasn't learned enough — try more epochs or a larger model.

In [30]:
print(metrics)

{'accuracy': 0.6078, 'precision': 0.6497, 'recall': 0.6078, 'f1': 0.591}


# Test and use the fine-tuned model

In [31]:
model = SetFitModel.from_pretrained('/content/setfit_2epochs')

preds = model.predict([
    'I cooked a delicious meal today!',
])
print('Prediction:', np.array(preds)[0])

Prediction: 1


C:\Users\joachim.querule\AppData\Local\Temp\ipykernel_16996\1539919560.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print('Prediction:', np.array(preds)[0])


In [32]:
preds = model.predict([
    'Some mornings are for sad songs and earl grey tea.',
])
print('Prediction:', np.array(preds)[0])

Prediction: 0


C:\Users\joachim.querule\AppData\Local\Temp\ipykernel_16996\341337659.py:4: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print('Prediction:', np.array(preds)[0])


In [33]:
preds = model.predict([
    "Texte à tester"
])

prediction = int(preds[0].item())

label_names = {
    0: "FALSE — tweet non réel",
    1: "TRUE — tweet réel"
}

print("Prédiction numérique :", prediction)
print("Résultat :", label_names[prediction])

Prédiction numérique : 0
Résultat : FALSE — tweet non réel


In [34]:
texte = "Texte à tester"

prediction = model.predict([texte])
probabilities = model.predict_proba([texte])

prediction_id = int(prediction[0].item())

print("Texte :", texte)
print("Prédiction :", prediction_id)
print("Probabilités :", probabilities[0])
print("Classe :", "Réel" if prediction_id == 1 else "Non réel")

Texte : Texte à tester
Prédiction : 0
Probabilités : tensor([0.6366, 0.3634])
Classe : Non réel
